<a href="https://colab.research.google.com/github/EnriqueCastilloRon/APLICACIONCURSOSFINAL/blob/main/Scientific_Dependency_Learning_v0_1_v0_2_COMPLETE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scientific Dependency Learning v0.1 + v0.2


In [2]:

# ==========================================================
# SCIENTIFIC DEPENDENCY LEARNING v0.1 + v0.2
# ==========================================================

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.interpolate import PchipInterpolator
import matplotlib.pyplot as plt

# ==========================================================
# DATASET
# ==========================================================

class DataSet:

    def __init__(self):
        self.df = None

    def load_csv(self, filepath):
        self.df = pd.read_csv(filepath)
        return self.df

    def summary(self):
        if self.df is None:
            print("No data loaded.")
            return
        print(self.df.describe(include="all"))

# ==========================================================
# EMPIRICAL DISTRIBUTION
# ==========================================================

class EmpiricalDistribution:

    def __init__(self):
        self.x_sorted = None
        self.u = None
        self.forward = None
        self.inverse = None

    def fit(self, x):

        x = np.asarray(x, dtype=float)
        x = np.sort(x)

        n = len(x)
        u = (np.arange(1, n + 1) - 0.5) / n

        xs, idx = np.unique(x, return_index=True)
        us = u[idx]

        self.x_sorted = xs
        self.u = us

        self.forward = PchipInterpolator(xs, us, extrapolate=True)
        self.inverse = PchipInterpolator(us, xs, extrapolate=True)

        return self

    def cdf(self, x):
        return np.clip(self.forward(x), 0.0, 1.0)

    def icdf(self, u):
        u = np.clip(u, 1e-12, 1 - 1e-12)
        return self.inverse(u)

    def sample(self, n):
        u = np.random.rand(n)
        return self.icdf(u)

# ==========================================================
# ECDF TRANSFORMER
# ==========================================================

class ECDFTransformer:

    def __init__(self):
        self.models = {}

    def fit(self, df):

        for c in df.columns:

            m = EmpiricalDistribution()
            m.fit(df[c].values)

            self.models[c] = m

        return self

    def transform(self, df):

        out = pd.DataFrame(index=df.index)

        for c in df.columns:
            out[c] = self.models[c].cdf(df[c].values)

        return out

    def inverse_transform(self, df_u):

        out = pd.DataFrame(index=df_u.index)

        for c in df_u.columns:
            out[c] = self.models[c].icdf(df_u[c].values)

        return out

# ==========================================================
# PI TRANSFORMER (v0.2)
# ==========================================================

class PiTransformer:

    def __init__(self):
        self.definitions = {}

    def add_variable(self, name, function):
        self.definitions[name] = function

    def transform(self, df):

        out = pd.DataFrame(index=df.index)

        for name, func in self.definitions.items():
            out[name] = func(df)

        return out


class FatiguePiTransformer(PiTransformer):

    def fit(self, sigmaM0, N0):

        self.sigmaM0 = sigmaM0
        self.N0 = N0

        self.add_variable(
            "pi1",
            lambda df: df["sigma_m"] / self.sigmaM0
        )

        self.add_variable(
            "pi2",
            lambda df: df["sigma_M"] / self.sigmaM0
        )

        self.add_variable(
            "pi3",
            lambda df: np.log10(df["N"] / self.N0)
        )

        return self

# ==========================================================
# MULTILINEAR BASIS
# ==========================================================

class MultilinearBasis:

    def __init__(self, max_order=2):

        self.max_order = max_order
        self.term_names = []

    def transform(self, df_u):

        cols = list(df_u.columns)

        X = []
        names = []

        for k in range(1, self.max_order + 1):

            for comb in combinations(range(len(cols)), k):

                term = np.ones(len(df_u))
                label = []

                for j in comb:

                    term *= df_u.iloc[:, j].values
                    label.append(cols[j])

                X.append(term)
                names.append("*".join(label))

        self.term_names = names

        return pd.DataFrame(
            np.column_stack(X),
            columns=names
        )

# ==========================================================
# LATENT MULTILINEAR MODEL
# ==========================================================

class LatentMultilinearModel:

    def __init__(self):

        self.coefficients = None
        self.term_names = None
        self.latent_values = None
        self.singular_values = None

    def fit(self, X_terms):

        X = X_terms.values

        U, S, VT = np.linalg.svd(
            X,
            full_matrices=False
        )

        w = VT[-1, :]

        w = w / np.linalg.norm(w)

        self.coefficients = w
        self.term_names = list(X_terms.columns)
        self.singular_values = S
        self.latent_values = X @ w

        return self

    def summary(self, top=20):

        print("\\nLATENT MULTILINEAR MODEL")
        print("=" * 40)

        print(
            f"Smallest singular value: {self.singular_values[-1]:.6e}"
        )

        idx = np.argsort(
            np.abs(self.coefficients)
        )[::-1]

        for i in idx[:top]:

            print(
                f"{self.term_names[i]:40s} "
                f"{self.coefficients[i]:12.6f}"
            )

    def plot_latent(self, bins=30):

        plt.figure(figsize=(7, 4))

        plt.hist(
            self.latent_values,
            bins=bins
        )

        plt.xlabel("Latent variable")
        plt.ylabel("Frequency")
        plt.title("Latent Variable Distribution")

        plt.show()


In [ ]:
# ==========================================
# EXAMPLE 1
# ==========================================

import numpy as np
import pandas as pd

np.random.seed(123)

n = 1000

df = pd.DataFrame({
    "X1": np.random.normal(0,1,n),
    "X2": np.random.gamma(2,2,n),
    "X3": np.random.weibull(2,n),
    "X4": np.random.uniform(-1,1,n)
})

ecdf = ECDFTransformer()

ecdf.fit(df)

df_u = ecdf.transform(df)

basis = MultilinearBasis(
    max_order=3
)

X_terms = basis.transform(df_u)

model = LatentMultilinearModel()

model.fit(X_terms)

model.summary()

model.plot_latent()

In [ ]:
# ==========================================
# EXAMPLE 2
# ==========================================

np.random.seed(123)

n = 1000

x1 = np.random.uniform(0,1,n)

x2 = np.random.uniform(0,1,n)

x3 = x1*x2 + 0.05*np.random.normal(size=n)

df = pd.DataFrame({
    "X1": x1,
    "X2": x2,
    "X3": x3
})

ecdf = ECDFTransformer().fit(df)

df_u = ecdf.transform(df)

basis = MultilinearBasis(
    max_order=2
)

X_terms = basis.transform(df_u)

model = LatentMultilinearModel()

model.fit(X_terms)

model.summary()

model.plot_latent()